# 🧠 Análisis Estadístico y Modelado: Sueño y Rendimiento Académico
**Actividad de Transferencia — Programación para Ciencia de Datos II**  
**Fundación Universitaria Compensar — Ingeniería en Ciencia de Datos**  

- **Autor:** Michael Marín Herrera (`mmarinh@ucompensar.edu.co`)  
- **Docente:** William Eduardo Clavijo Bohorquez  

---
### 🎯 Objetivo General
Evaluar el impacto de los hábitos de sueño (horas dormidas), tiempo de estudio y nivel de estrés sobre el rendimiento académico (puntaje en examen de 0 a 100), mediante técnicas de:
1. Análisis Exploratorio de Datos (EDA)
2. Pruebas de Contraste de Hipótesis (Welch $t$-test & Mann-Whitney $U$)
3. Regresión Lineal Múltiple con métricas de bondad de ajuste
4. Regresión Logística y Diagnóstico de Riesgo Académico

In [ ]:
# 1. Importación de Librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix, classification_report

print("✅ Librerías cargadas exitosamente.")

## 📥 2. Carga y Exploración Inicial del Dataset

In [ ]:
# Cargar dataset
try:
    df = pd.read_csv('data/sueno_rendimiento.csv')
except FileNotFoundError:
    df = pd.read_csv('sueno_rendimiento.csv')

display(df.head(10))
print("\nResumen estadístico básico:")
display(df.describe())

## 🔬 3. Contraste de Hipótesis: Sueño vs. Rendimiento
Planteamiento:
- $H_0$: $\mu_{\text{alto sueño}} \le \mu_{\text{bajo sueño}}$ (Dormir más no incrementa el puntaje promedio)
- $H_1$: $\mu_{\text{alto sueño}} > \mu_{\text{bajo sueño}}$ (Dormir más incrementa significativamente el puntaje promedio)

In [ ]:
corte_sueno = df['horas_sueno'].quantile(0.50)
grupo_alto = df[df['horas_sueno'] >= corte_sueno]['puntaje']
grupo_bajo = df[df['horas_sueno'] < corte_sueno]['puntaje']

t_stat, p_val_two_sided = stats.ttest_ind(grupo_alto, grupo_bajo, equal_var=False)
p_val_welch = p_val_two_sided / 2 if t_stat > 0 else 1.0 - (p_val_two_sided / 2)

u_stat, p_val_mwu_two = stats.mannwhitneyu(grupo_alto, grupo_bajo, alternative='greater')

print(f"Corte mediana de sueño: {corte_sueno:.2f} hrs")
print(f"Prueba t de Welch: t = {t_stat:.4f}, p-valor (unilateral) = {p_val_welch:.6f}")
print(f"Prueba Mann-Whitney U: U = {u_stat:.2f}, p-valor = {p_val_mwu_two:.6f}")

if p_val_welch < 0.05:
    print("🎉 Conclusión: Se rechaza H0 a favor de H1 (p < 0.05). Existe evidencia estadística suficiente.")
else:
    print("⚠️ Conclusión: No se rechaza H0 (p >= 0.05).")

## 📈 4. Regresión Lineal Múltiple
Modelamos el puntaje esperado en función de las variables predictoras.

In [ ]:
features = ['horas_sueno', 'horas_estudio', 'nivel_estres']
X = df[features]
y = df['puntaje']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

modelo_lin = LinearRegression()
modelo_lin.fit(X_train, y_train)

y_pred = modelo_lin.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Coeficiente R² en test: {r2:.4f}")
print(f"RMSE en test: {rmse:.4f} pts")
print("\nCoeficientes del modelo:")
for feat, coef in zip(features, modelo_lin.coef_):
    print(f" - {feat}: {coef:+.4f}")
print(f" - Intercepto (b0): {modelo_lin.intercept_:.4f}")

## 🩺 5. Regresión Logística y Detección de Riesgo Académico

In [ ]:
umbral_puntaje = df['puntaje'].median()
df['en_riesgo'] = (df['puntaje'] <= umbral_puntaje).astype(int)

X_log = df[['horas_sueno', 'horas_estudio', 'nivel_estres']]
y_log = df['en_riesgo']

modelo_log = LogisticRegression(C=1.0, random_state=42)
modelo_log.fit(X_log, y_log)

probs = modelo_log.predict_proba(X_log)[:, 1]
preds = (probs >= 0.5).astype(int)

print(f"Precisión global (Accuracy): {accuracy_score(y_log, preds):.2%}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_log, preds))
print("\nReporte de Clasificación:")
print(classification_report(y_log, preds))

## 🚀 6. Lanzar la Aplicación Streamlit en Binder / Colab
Ejecuta la siguiente celda si deseas levantar el dashboard interactivo web dentro del entorno de Binder o Jupyter.

In [ ]:
# Para ejecutar localmente o en terminal:
# streamlit run dashboard_sueno_rendimiento.py
print("Para iniciar el dashboard interactivo, ejecuta en la terminal:")
print("streamlit run dashboard_sueno_rendimiento.py")